In [1]:
from utils.metrics import mse_loss, dice_loss, precision_recall_f1, pixel_accuracy, point_matching, visualize_batch_matching  

In [ ]:
# Now checking for experiments 

# Check with images from Experiment to seed matching, using vae 

from utils.config import OUTPUT_DIR_EXPTOSEED_COLOR_SAVED, SEED_FOLDER_EXPTOSIM_TEST_FIXED_NONFIXED_32X32
import os 
import torch
import cv2
import numpy as np
from utils.preprocess import preprocess_seed

prediction_files= sorted(os.listdir(OUTPUT_DIR_EXPTOSEED_COLOR_SAVED))
gt_files=  sorted(os.listdir(SEED_FOLDER_EXPTOSIM_TEST_FIXED_NONFIXED_32X32))


# prediction files has output prefix, and gt files has input prefix, apart from that they match

pred_img_tensor_list= []
gt_img_tensor_list= []

mse_loss_list= []
dice_loss_list= []
precision_recall_f1_list =[]
pixel_accuracy_list =[]

precision_rtol_list= []
accuracy_rtol_list= []
f1_rtol_list= []


for pred_file, gt_file in zip(prediction_files, gt_files):

    
    pred_img= cv2.imread(os.path.join(OUTPUT_DIR_EXPTOSEED_COLOR_SAVED, pred_file), cv2.IMREAD_GRAYSCALE)
    # Apply same preprocessing as training dataset to match pixel alignment
    gt_img= preprocess_seed(os.path.join(SEED_FOLDER_EXPTOSIM_TEST_FIXED_NONFIXED_32X32, gt_file), img_length=32, img_width=32)


    pred_img_tensor= torch.from_numpy(pred_img)
    gt_img_tensor= torch.from_numpy(gt_img)

    pred_img_tensor_list.append(pred_img_tensor)
    gt_img_tensor_list.append(gt_img_tensor)



    precision, recall, f1_score = point_matching((pred_img_tensor),(gt_img_tensor), tolerance_radius=4, algorithm='hungarian')

    precision_rtol_list.append(precision)
    accuracy_rtol_list.append(recall)
    f1_rtol_list.append(f1_score)
    
    
    mse_value = mse_loss(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    dice_value = dice_loss(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    precision, recall, f1_score = precision_recall_f1(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    pixel_acc = pixel_accuracy(torch.from_numpy(pred_img), torch.from_numpy(gt_img))




    mse_loss_list.append(mse_value.item())
    dice_loss_list.append(dice_value.item())
    precision_recall_f1_list.append((precision, recall, f1_score))
    pixel_accuracy_list.append(pixel_acc)

    

In [3]:
# Experiments 

precision_avg = np.mean([pr[0] for pr in precision_recall_f1_list])
recall_avg = np.mean([pr[1] for pr in precision_recall_f1_list])
f1_avg = np.mean([pr[2] for pr in precision_recall_f1_list])

precision_std= np.std([pr[0] for pr in precision_recall_f1_list])
recall_std= np.std([pr[1] for pr in precision_recall_f1_list])
f1_std= np.std([pr[2] for pr in precision_recall_f1_list])  


print(f"MSE Mean: {np.mean(mse_loss_list)} Std: {np.std(mse_loss_list)}")
print(f"DICE loss Mean: {np.mean(dice_loss_list)} Std :{np.std(dice_loss_list)}")
print(f"Precision : {precision_avg:.3f} ± {precision_std:.3f}")
print(f"Recall : {recall_avg:.3f} ± {recall_std:.3f}")
print(f"F1 Score : {f1_avg:.3f} ± {f1_std:.3f}")
print(f"Pixel Accuracy average: {np.mean(pixel_accuracy_list):.3f} ± {np.std(pixel_accuracy_list):.3f}")



MSE Mean: 0.016142874053030304 Std: 0.01165968644483593
DICE loss Mean: 0.9132591341481064 Std :0.14006727997821092
Precision : 0.100 ± 0.155
Recall : 0.084 ± 0.142
F1 Score : 0.087 ± 0.140
Pixel Accuracy average: 0.984 ± 0.012


In [ ]:
# Experiments, tolerance point matching results (radius=4, Hungarian)

print(f"Tolerance Precision {np.mean(precision_rtol_list)} ± {np.std(precision_rtol_list)}")
print(f"Tolerance Recall {np.mean(accuracy_rtol_list)} ± {np.std(accuracy_rtol_list)}")
print(f"Tolerance F1 {np.mean(f1_rtol_list)} ± {np.std(f1_rtol_list)}")

Tolerance Precision 0.7985216861802472 ± 0.20583826235573333
Tolerance Recall 0.7200498247403343 ± 0.26629361336276564
Tolerance F1 0.723321132922292 ± 0.20771184874643703
